# Within-language distance tests (Tables 3, 4, D.2, D.3)

For each conflict, judge and judge prompt, computes mean within-language similarity `s = 1 - d` per model / prompt-group cell, plus paired t-test and BH-FDR adjusted p-values.

In [ ]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel

from src.conflicts import get_conflict
from src.statistics.aggregation import bh_fdr, load_run_scores, paired


## Config


In [ ]:
CONFLICT = 'ru_ua'
RESULTS_ROOT = '../../results'
JUDGE_LABEL = 'llama3-8b'
JUDGE_PROMPT = 'primary'

MODELS = [
    'mistral-7b-instruct-v0.3',
    'qwen25-7b',
    'llama3-8b',
    'gemini-3-pro-preview',
    'deepseek-7b-chat',
    'falcon-7b-instruct',
    'moonshot-v1-8k',
    'gpt-5.1',
]

ALPHA = 0.05


## Prompt groups


In [ ]:
PROMPT_GROUPS = {
    'baseline': ['neutral_1', 'neutral_2', 'neutral_3'],
    'social_influence': ['influencer_1', 'influencer_2', 'twitter_1', 'twitter_2', 'social_media_reporter_1', 'social_media_reporter_2'],
    'news': ['telegram_channel_1', 'telegram_channel_2', 'news_article_1', 'news_article_2'],
    'journalist_voice': ['social_media_reporter_1', 'social_media_reporter_2', 'telegram_channel_1', 'news_article_1', 'news_article_2'],
    'country_aligned': ['telegram_channel_1', 'telegram_channel_2', 'twitter_1', 'twitter_2', 'influencer_1', 'influencer_2', 'news_article_1'],
}


## Collect per-cell paired scores and compute tests


In [ ]:
conflict = get_conflict(CONFLICT)
side_a, side_b = conflict.sides

def cell_scores(model, group_templates):
    a_all, b_all = {}, {}
    for prompt in group_templates:
        a = load_run_scores(results_root=RESULTS_ROOT, conflict=conflict, judge_label=JUDGE_LABEL, judge_prompt=JUDGE_PROMPT, translator_label=model, prompt_name=prompt, side=side_a)
        b = load_run_scores(results_root=RESULTS_ROOT, conflict=conflict, judge_label=JUDGE_LABEL, judge_prompt=JUDGE_PROMPT, translator_label=model, prompt_name=prompt, side=side_b)
        for k, v in a.items():
            a_all[f'{prompt}/{k}'] = v
        for k, v in b.items():
            b_all[f'{prompt}/{k}'] = v
    return a_all, b_all

rows = []
for model in MODELS:
    for group, templates in PROMPT_GROUPS.items():
        a, b = cell_scores(model, templates)
        xa, xb = paired(a, b)
        if len(xa) == 0:
            rows.append({'model': model, 'group': group, 'n': 0, f'mean_sim_{side_a.code}': None, f'mean_sim_{side_b.code}': None, 't': None, 'p': None})
            continue
        sim_a = 1.0 - xa
        sim_b = 1.0 - xb
        t = ttest_rel(xa, xb)
        rows.append({
            'model': model,
            'group': group,
            'n': len(xa),
            f'mean_sim_{side_a.code}': float(sim_a.mean()),
            f'mean_sim_{side_b.code}': float(sim_b.mean()),
            f'mean_dist_{side_a.code}': float(xa.mean()),
            f'mean_dist_{side_b.code}': float(xb.mean()),
            'diff_a_minus_b': float((xa - xb).mean()),
            't': float(t.statistic),
            'p': float(t.pvalue),
        })

df = pd.DataFrame(rows)
finite = df['p'].notna()
if finite.any():
    df.loc[finite, 'q_bh'] = bh_fdr(df.loc[finite, 'p'].tolist())
    df['sig'] = df['q_bh'] < ALPHA
df


## Similarity table (Table 3 / Table 4 layout)


In [ ]:
def fmt_sim(row):
    a = row.get(f'mean_sim_{side_a.code}')
    b = row.get(f'mean_sim_{side_b.code}')
    if a is None or b is None:
        return '-'
    star = '*' if row.get('sig') else ''
    left = f'{a:.3f}'
    right = f'{b:.3f}'
    if a > b:
        left = f'**{left}**'
    elif b > a:
        right = f'**{right}**'
    return f'{left}/{right}{star}'

sim_tbl = pd.DataFrame({m: {g: fmt_sim(df.query('model == @m and group == @g').iloc[0]) for g in PROMPT_GROUPS} for m in MODELS})
sim_tbl.index.name = f'{side_a.code.upper()} / {side_b.code.upper()} similarity'
sim_tbl


## Raw distance table (Table D.2 / D.3 layout)


In [ ]:
def fmt_dist(row):
    a = row.get(f'mean_dist_{side_a.code}')
    b = row.get(f'mean_dist_{side_b.code}')
    if a is None or b is None:
        return '-'
    star = '*' if row.get('sig') else ''
    left = f'{a:.3f}'
    right = f'{b:.3f}'
    if a < b:
        left = f'**{left}**'
    elif b < a:
        right = f'**{right}**'
    return f'{left}/{right}{star}'

dist_tbl = pd.DataFrame({m: {g: fmt_dist(df.query('model == @m and group == @g').iloc[0]) for g in PROMPT_GROUPS} for m in MODELS})
dist_tbl.index.name = f'{side_a.code.upper()} / {side_b.code.upper()} distance'
dist_tbl
